In [ ]:
import sys
import pandas as pd
pd.options.display.max_colwidth = 600

from IPython.display import IFrame, display

sys.path.append('/global/homes/b/bkieft/metatlas2/metatlas2')
import logging_config as lcf
import logging
import workflow_objects as wfo
import load_tools as ldt

In [ ]:
# Set up logging with level
log_level = logging.INFO
lcf.setup_logging(log_level=log_level)
logger = lcf.get_logger('analysis_gui', log_level=log_level)

In [ ]:
# Load configuration files
ANALYSIS_CONFIG = ldt.load_metatlas2_config("/global/homes/b/bkieft/metatlas2/configs/analysis.yaml")

In [ ]:
# Set up analysis parameters
PROJECT_NAME = "20260106_JGI_LP-AC_510770_StrigoArab_final_EXP120B_HILICZ_USHXG03402"
RT_ALIGN_NUM = 0
ANALYSIS_NUM = 0

In [ ]:
import os, sys, threading, time
from IPython.display import display, HTML

PORT = 8050  # ← single source of truth

# Reload modules to avoid stale Flask state
for mod in list(sys.modules.keys()):
    if 'analysis_gui' in mod or 'wfo' in mod:
        del sys.modules[mod]

dash_app = wfo.run_analysis_gui(
    config=ANALYSIS_CONFIG, 
    project_name=PROJECT_NAME, 
    rt_alignment_number=RT_ALIGN_NUM, 
    analysis_number=ANALYSIS_NUM,
    dash_app_port=PORT
)

def _run():
    dash_app.run(
        host="0.0.0.0",
        port=PORT,           # ← same port
        debug=False,
        use_reloader=False,
        dev_tools_hot_reload=False,
    )

t = threading.Thread(target=_run, daemon=True)
t.start()
time.sleep(3)

service_prefix = os.getenv('JUPYTERHUB_SERVICE_PREFIX', '/')
url = f"{service_prefix}proxy/{PORT}/"
print(f"App running at: {url}")
display(HTML(f'<a href="{url}" target="_blank">▶ Open Dash App ↗</a>'))